<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vsep — Audio Stem Separator

Separate any audio into stems (vocals, drums, bass, etc.) using state-of-the-art AI models.

---

In [ ]:
#@title 1. Install
#@markdown Clone repo, install deps, yt-dlp, ffmpeg

!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep
!pip install -q -r /content/vsep/requirements.txt yt-dlp
!apt-get -qq install ffmpeg > /dev/null 2>&1

import os; os.chdir('/content/vsep')
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU — go to Runtime > Change runtime type")
import yt_dlp; print(f"yt-dlp {yt_dlp.version.__version__} ready")

In [ ]:
#@title 2. Get Audio
#@markdown Upload a file OR paste a YouTube/SoundCloud URL

import os, glob
audio_source = "Upload file" #@param ["Upload file", "YouTube / URL"]

if audio_source == "Upload file":
    from google.colab import files
    print("Upload your audio (MP3, WAV, FLAC, OGG, M4A)...")
    uploaded = files.upload()
    if uploaded:
        audio_file = list(uploaded.keys())[0]
    else:
        raise SystemExit("No file uploaded.")
else:
    url = "" #@param {type:"string"}
    if not url.strip():
        raise SystemExit("Paste a URL first.")
    os.makedirs("/content/vsep/audio_input", exist_ok=True)
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": "/content/vsep/audio_input/%(title).80s.%(ext)s",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav", "preferredquality": "0"}],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url.strip()])
    audio_file = sorted(glob.glob("/content/vsep/audio_input/*.wav"))[-1]

print(f"\nAudio: {audio_file} ({os.path.getsize(audio_file)/1e6:.1f} MB)")

In [ ]:
#@title 3. Select Model
#@markdown Pick a model. Display name is mapped to the actual filename automatically.

MODEL_CATALOG = {
    "BS Roformer EP 317 SDR 12.98":      "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
    "Mel Roformer Viperx SDR 12.61":     "Mel-Roformer-Viperx-1053.ckpt",
    "BS Roformer Viperx 1297":           "BS-Roformer-Viperx-1297.ckpt",
    "MDX23C InstVoc HQ SDR 11.95":      "MDX23C-8KFFT-InstVoc_HQ.ckpt",
    "Demucs v4 Fine Tuned SDR 11.27":    "ht-demucs_ft.yaml",
    "Reverb HQ SDR 11.20":              "MDX23C-8KFFT-Reverb_HQ.ckpt",
    "Echo HQ SDR 11.10":                "MDX23C-8KFFT-Echo_HQ.ckpt",
    "De-Esser MDX SDR 10.90":           "MDX23C-8KFFT-DeEsser.ckpt",
    "MDX Net Inst HQ 2 SDR 10.93":      "UVR-MDX-NET-Inst_HQ_2.onnx",
    "Guitar Isolation SDR 10.80":       "MDX23C-8KFFT-Guitar.ckpt",
    "MDX23C InstVoc SDR 10.75":         "MDX23C-8KFFT-InstVoc.ckpt",
    "Drum Isolation SDR 10.50":         "MDX23C-8KFFT-Drum.ckpt",
    "MDX Net Inst 1 SDR 10.65":         "UVR-MDX-NET-Inst_1.onnx",
    "Bass Isolation SDR 10.10":         "MDX23C-8KFFT-Bass.ckpt",
    "MDX Net Inst HQ 4 SDR 10.49":      "UVR-MDX-NET-Inst_HQ_4.onnx",
    "Piano Isolation SDR 9.90":         "MDX23C-8KFFT-Piano.ckpt",
    "Kim Vocal 2":                      "Kim_Vocal_2.onnx",
    "MDX Net Karaoke 2":                "UVR-MDX-NET-KARA_2.onnx",
    "Demucs v4 6 Stem":                 "htdemucs_6s.yaml",
    "VR 5 HP":                          "5_HP-UVR.pth",
}

model = "BS Roformer EP 317 SDR 12.98" #@param ["BS Roformer EP 317 SDR 12.98", "Mel Roformer Viperx SDR 12.61", "BS Roformer Viperx 1297", "MDX23C InstVoc HQ SDR 11.95", "Demucs v4 Fine Tuned SDR 11.27", "Reverb HQ SDR 11.20", "Echo HQ SDR 11.10", "De-Esser MDX SDR 10.90", "MDX Net Inst HQ 2 SDR 10.93", "Guitar Isolation SDR 10.80", "MDX23C InstVoc SDR 10.75", "Drum Isolation SDR 10.50", "MDX Net Inst 1 SDR 10.65", "Bass Isolation SDR 10.10", "MDX Net Inst HQ 4 SDR 10.49", "Piano Isolation SDR 9.90", "Kim Vocal 2", "MDX Net Karaoke 2", "Demucs v4 6 Stem", "VR 5 HP"]
selected_model = MODEL_CATALOG[model]
print(f"Model: {model}")
print(f"File:  {selected_model}")

In [ ]:
#@title 4. Separate
#@markdown Run the AI separation on your audio

from separator import Separator
import config.variables as cfg

cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print(f"Separating {audio_file} with {selected_model}...")
print("(downloading model on first use, cached after)")

separator = Separator(
    model_file_dir="/content/vsep/models",
    output_dir="/content/vsep/output",
    output_format="WAV",
    use_soundfile=True,
)
separator.load_model(model_filename=selected_model)
output_files = separator.separate(audio_file)

print(f"\nDone! {len(output_files)} stem(s):")
for f in output_files:
    print(f"  {f}")

In [ ]:
#@title 5. Listen & Download
#@markdown Play the results and download to your device

import glob
from IPython.display import Audio, display
from google.colab import files as dl

stems = sorted(glob.glob("/content/vsep/output/*.*"))
if not stems:
    raise SystemExit("No stems found. Run the separation cell first.")

for i, path in enumerate(stems, 1):
    name = os.path.basename(path)
    size = os.path.getsize(path) / 1e6
    print(f"{i}. {name} ({size:.1f} MB)")
    display(Audio(path))

print("\n--- Downloading all stems ---")
for path in stems:
    dl.download(path)